In [ ]:
# --- Deep learning framework ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import Adam
from einops import rearrange

# --- Data processing ---
import os
import numpy as np
import pandas as pd
from PIL import ImageFile, Image
Image.MAX_IMAGE_PIXELS = 100000000

# --- Scientific computing ---
from scipy.spatial import distance
import scipy.stats as st
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

# --- Spatial transcriptomics ---
import scanpy as sc
import scprep as scp

# --- Pretrained models ---
from transformers import AutoImageProcessor, AutoModel

# --- Custom ViT module ---
from finetune.vision_transformer import vit_small

In [ ]:
# --- Training configuration ---
fold = 5                 # LOO-CV fold index
tag = '-test'
learning_rate = 1e-5
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## 1. GPU-Accelerated Pearson Correlation

In [ ]:
def cal_Percor_gpu(true, pred, axis=1):
    """
    GPU-accelerated Pearson correlation using PyTorch.
    Pearson r = cosine similarity of centered vectors.

    Args:
        true, pred: gene expression tensors
        axis: 1 = per-spot across genes, 0 = per-gene across spots
    Returns:
        corr: correlation vector, mean_corr: scalar average
    """
    if axis == 0:
        true = true.T
        pred = pred.T

    true_mean = torch.mean(true, dim=1, keepdim=True)
    pred_mean = torch.mean(pred, dim=1, keepdim=True)
    true_centered = true - true_mean
    pred_centered = pred - pred_mean

    corr = torch.nn.functional.cosine_similarity(true_centered, pred_centered, dim=1)
    return corr, torch.nanmean(corr)


def predict_and_evaluate(model, test_dataloader, device):
    """Evaluate on test set, keeping all computation on GPU."""
    model.eval()
    all_corr1, all_corr2 = [], []

    with torch.no_grad():
        for patch, feature, exp, adj, center, position in test_dataloader:
            patch, feature, exp, adj, center, position = [
                x.to(device) for x in [patch, feature, exp, adj, center, position]
            ]
            exp = exp.squeeze()
            recon_gene = model(patch, feature, adj, center, position)
            _, m1 = cal_Percor_gpu(exp, recon_gene, axis=1)  # Spot-level
            _, m2 = cal_Percor_gpu(exp, recon_gene, axis=0)  # Gene-level
            all_corr1.append(m1.item())
            all_corr2.append(m2.item())

    return np.mean(all_corr1), np.mean(all_corr2)

## 2. Her2ST Dataset

Loads H&E patches, UNI2 features, gene expression, and builds an image-feature-weighted spatial adjacency graph for GAT.

In [ ]:
class Her2ST(torch.utils.data.Dataset):
    """her2st spatial transcriptomics dataset with leave-one-out CV split."""

    def __init__(self, train=True, fold=0):
        super(Her2ST, self).__init__()

        self.exp_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-cnts'
        self.img_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-imgs'
        self.pos_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-spotfiles'
        self.feature_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-UNI2-features'
        self.patch_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-patches'
        self.gene_list = list(np.load(
            r'./对比方法/THItoGene-main/data/her_hvg_cut_1000.npy', allow_pickle=True))
        self.r = 56

        names = [file for file in os.listdir(self.exp_dir) if file.endswith('.tsv')]
        names.sort()
        names = [i[:2] for i in names]
        names = [item for item in names if item not in ['A1', 'H1', 'H2', 'H3']]
        self.train = train

        # Leave-one-out split: 1 test sample, rest for training
        te_names = [names[fold]]
        tr_names = list(set(names) - set(te_names))
        self.names = tr_names if train else te_names

        print('Loading imgs...')
        self.img_dict = {i: torch.Tensor(np.array(self.get_img(i))) for i in self.names}
        print('Loading expdata...')
        self.meta_dict = {i: self.get_meta(i) for i in self.names}
        self.feature_dict = {i: self.get_features(i) for i in self.names}
        self.patch_dict = {i: self.get_patches(i) for i in self.names}
        self.exp_dict = {key: self.preprocess(data) for key, data in self.meta_dict.items()}
        self.adj_dict = {
            key: self.construct_knn_image_graph(data, img_feat, k=4, min_degree=2)
            for (key, data), (_, img_feat) in zip(self.exp_dict.items(), self.feature_dict.items())
        }

    def __getitem__(self, index):
        name = self.names[index]
        exps = self.exp_dict[name].X
        features = self.feature_dict[name]
        centers = (np.floor(self.exp_dict[name].obs[['pixel_x', 'pixel_y']]).values).astype(int)
        positions = (np.floor(self.exp_dict[name].obs[['x', 'y']]).values).astype(int)
        patches = self.patch_dict[name]
        adj = self.adj_dict[name]
        return patches, features, torch.Tensor(exps), adj, torch.Tensor(centers), torch.Tensor(positions)

    def __len__(self):
        return len(self.exp_dict)

    def preprocess(self, adata):
        """Filter HVGs, normalize to 10K counts, log1p."""
        adata = adata[:, self.gene_list]
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        return adata

    def get_img(self, name):
        pre = self.img_dir + '/' + name[0] + '/' + name
        fig_name = [file for file in os.listdir(pre) if file.endswith('.jpg')][0]
        path = pre + '/' + fig_name
        print(path)
        return Image.open(path)

    def get_meta(self, name):
        exp_path = self.exp_dir + '/' + name + '.tsv'
        adata = sc.read(exp_path, delimiter='\t')
        pos_path = self.pos_dir + '/' + name + '_selection.tsv'
        df = pd.read_csv(pos_path, sep='\t')
        x = np.around(df['x'].values).astype(int)
        y = np.around(df['y'].values).astype(int)
        id = [str(x[i]) + 'x' + str(y[i]) for i in range(len(x))]
        df.index = id
        adata.obs = adata.obs.merge(df, how='left', left_index=True, right_index=True)
        return adata

    def get_features(self, name):
        feature_path = self.feature_dir + '/' + name + '.pt'
        return torch.load(feature_path)

    def get_patches(self, name):
        patch_path = self.patch_dir + '/' + name + '.pt'
        return torch.load(patch_path)

    def construct_knn_image_graph(self, adata, img_feat, k=4, min_degree=2, eps=1e-6):
        """
        Build a weighted adjacency graph combining spatial kNN with image feature similarity.
        Steps: spatial kNN → cosine similarity of image features → fuse → ensure min_degree → symmetrize.
        """
        pos = adata.obs[['pixel_x', 'pixel_y']].values.astype(float)
        N = pos.shape[0]

        nbrs = NearestNeighbors(n_neighbors=k + 1).fit(pos)
        _, indices = nbrs.kneighbors(pos)
        knn_idx = indices[:, 1:]  # Exclude self

        adj_spatial = np.zeros((N, N), dtype=np.float32)
        for i in range(N):
            adj_spatial[i, knn_idx[i]] = 1.0
        adj_spatial = np.maximum(adj_spatial, adj_spatial.T)

        sim = cosine_similarity(img_feat)
        sim = np.clip(sim, 0.0, 1.0)
        adj = adj_spatial * sim

        for i in range(N):
            current_degree = int((adj[i] > eps).sum())
            if current_degree < min_degree:
                num_needed = min_degree - current_degree
                dist = np.linalg.norm(pos - pos[i], axis=1)
                nearest_idxs = np.argsort(dist)[1:]
                added = 0
                for j in nearest_idxs:
                    if adj[i, j] <= eps:
                        adj[i, j] = sim[i, j]
                        adj[j, i] = sim[j, i]
                        added += 1
                    if added >= num_needed:
                        break

        adj = (adj + adj.T) / 2.0
        return torch.tensor(adj, dtype=torch.float32)

## 3. Model Components

### 3.1 ConvNeXtBlock
Depthwise conv → LayerNorm → SwiGLU MLP → LayerScale → residual.

In [ ]:
class ConvNeXtBlock(nn.Module):
    """ConvNeXt basic block with 7×7 depthwise conv, LayerNorm, SwiGLU MLP, and LayerScale."""
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(torch.zeros(dim))  # LayerScale

    def forward(self, x):
        shortcut = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)           # [B,C,H,W] → [B,H,W,C]
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2)           # [B,H,W,C] → [B,C,H,W]
        return x + shortcut

### 3.2 GatedFusion
Dynamically weights three branches (3×3 / 5×5 / 7×7) via global pooling + FC + softmax.

In [ ]:
class GatedFusion(nn.Module):
    """Learn per-branch contribution weights for multi-scale feature fusion."""
    def __init__(self, dim):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(dim * 3, dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(dim // 2, 3),
            nn.Softmax(dim=1)
        )

    def forward(self, x1, x2, x3):
        b, c, _, _ = x1.size()
        cat_feat = torch.cat([self.avg_pool(x1), self.avg_pool(x2), self.avg_pool(x3)], dim=1).view(b, -1)
        weights = self.fc(cat_feat).view(b, 3, 1, 1, 1)  # [B, 3, 1, 1, 1]
        return x1 * weights[:, 0] + x2 * weights[:, 1] + x3 * weights[:, 2]

### 3.3 ImprovedMultiScaleConvNeXt
Three-branch parallel convolutions with different kernel sizes → gated fusion → post-processing.

In [ ]:
class ImprovedMultiScaleConvNeXt(nn.Module):
    """Multi-scale feature extractor: 3×3 / 5×5 / 7×7 branches → gated fusion."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=7, padding=1),
            ConvNeXtBlock(out_channels))
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=5, stride=7, padding=2),
            ConvNeXtBlock(out_channels))
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=7, stride=7, padding=3),
            ConvNeXtBlock(out_channels))
        self.fusion = GatedFusion(out_channels)
        self.post_conv = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU())

    def forward(self, x):
        out = self.fusion(self.branch1(x), self.branch2(x), self.branch3(x))
        return self.post_conv(out)

### 3.4 ViT (ConvNeXt + FFN Transformer)
Attention-free transformer: ConvNeXt → pooling → position encoding → FFN layers with residuals.

In [ ]:
class ConvNeXt_FFN_Transformer(nn.Module):
    """FFN-only transformer — replaces self-attention with feed-forward layers."""
    def __init__(self, channel=32, dim=1024, depth1=2, depth2=4, dropout=0.1):
        super().__init__()
        self.convnext_layers = nn.Sequential(
            *[ConvNeXtBlock(dim=channel) for _ in range(depth1)])
        self.down = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(channel, dim), nn.LayerNorm(dim))
        self.ffn_layers = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(dim), nn.Linear(dim, dim * 2), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(dim * 2, dim), nn.Dropout(dropout))
            for _ in range(depth2)])

    def forward(self, x, ct):
        x = self.convnext_layers(x)         # [B*N, 32, H, W]
        x = self.down(x)                    # [B*N, 1024]
        g = x.unsqueeze(0) + ct             # Inject position encoding
        for ffn in self.ffn_layers:
            g = g + ffn(g)                  # Pre-LN residual
        return g.squeeze(0)                 # [N, 1024]


class ViT(nn.Module):
    """ViT wrapper with input dropout."""
    def __init__(self, channel=32, dim=1024, depth1=2, depth2=4, dropout=0.):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.model = ConvNeXt_FFN_Transformer(
            channel=channel, dim=dim, depth1=depth1, depth2=depth2, dropout=dropout)

    def forward(self, x, ct):
        return self.model(self.dropout(x), ct)

### 3.5 Multi-Head Graph Attention Network (GAT)

In [ ]:
class GraphAttentionLayer(nn.Module):
    """Single-head GAT layer (Veličković et al., 2018)."""
    def __init__(self, in_features, out_features, dropout=0.2, alpha=0.01, concat=True):
        super(GraphAttentionLayer, self).__init__()
        self.dropout = dropout
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        self.concat = concat
        self.W = nn.Parameter(torch.empty(size=(in_features, out_features)))
        nn.init.xavier_uniform_(self.W.data, gain=1.414)
        self.a = nn.Parameter(torch.empty(size=(2 * out_features, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.414)
        self.leakyrelu = nn.LeakyReLU(self.alpha)

    def forward(self, h, adj):
        Wh = torch.mm(h, self.W)
        e = self._prepare_attentional_mechanism_input(Wh)
        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)  # Mask non-neighbors
        attention = F.softmax(attention, dim=1)
        attention = F.dropout(attention, self.dropout, training=self.training)
        h_prime = torch.matmul(attention, Wh)
        return F.elu(h_prime) if self.concat else h_prime

    def _prepare_attentional_mechanism_input(self, Wh):
        Wh1 = torch.matmul(Wh, self.a[:self.out_features, :])
        Wh2 = torch.matmul(Wh, self.a[self.out_features:, :])
        return self.leakyrelu(Wh1 + Wh2.T)


class MultiHeadGAT(nn.Module):
    """Multi-head GAT: first layer concatenates heads, second layer averages."""
    def __init__(self, in_features, nhid, out_features, dropout, alpha, heads=4):
        super(MultiHeadGAT, self).__init__()
        self.dropout = dropout
        self.attentions = [
            GraphAttentionLayer(in_features, nhid, dropout=dropout, alpha=alpha, concat=True)
            for _ in range(heads)]
        for i, attention in enumerate(self.attentions):
            self.add_module('attention_{}'.format(i), attention)
        self.out_att = GraphAttentionLayer(
            nhid * heads, out_features, dropout=dropout, alpha=alpha, concat=False)

    def forward(self, x, adj):
        x = F.dropout(x, self.dropout, training=self.training)
        x = torch.cat([att(x, adj).squeeze(0) for att in self.attentions], dim=1)
        x = F.dropout(x, self.dropout, training=self.training)
        return F.elu(self.out_att(x, adj))

### 3.6 AttentionLayer — Multi-modal Fusion
Adaptively weights two feature modalities via learned attention.

In [ ]:
class AttentionLayer(nn.Module):
    """Learned attention-weighted fusion of two embedding modalities."""
    def __init__(self, in_feat, out_feat, dropout=0.0, act=F.relu):
        super(AttentionLayer, self).__init__()
        self.in_feat = in_feat
        self.out_feat = out_feat
        self.w_omega = nn.Parameter(torch.FloatTensor(in_feat, out_feat))
        self.u_omega = nn.Parameter(torch.FloatTensor(out_feat, 1))
        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.w_omega)
        torch.nn.init.xavier_uniform_(self.u_omega)

    def forward(self, emb1, emb2):
        emb = []
        emb.append(torch.unsqueeze(torch.squeeze(emb1), dim=1))
        emb.append(torch.unsqueeze(torch.squeeze(emb2), dim=1))
        self.emb = torch.cat(emb, dim=1)                     # [N, 2, D]
        self.v = F.tanh(torch.matmul(self.emb, self.w_omega))
        self.vu = torch.matmul(self.v, self.u_omega)
        self.alpha = F.softmax(torch.squeeze(self.vu) + 1e-6, dim=1)  # [N, 2]
        a = torch.transpose(self.emb, 1, 2)                  # [N, D, 2]
        b = torch.unsqueeze(self.alpha, -1)                  # [N, 2, 1]
        emb_combined = torch.matmul(a, b)                    # [N, D, 1]
        return torch.squeeze(emb_combined)                   # [N, D]

### 3.7 GlobalToLocalCrossAttention
Injects global slide-level context into local spot features: spot_feat (Q) attends to global_feat (K, V).

In [ ]:
class GlobalToLocalCrossAttention(nn.Module):
    """Cross-attention: spot features (query) attend to global slide embedding (key/value)."""
    def __init__(self, dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads,
                                           dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, spot_feat, global_feat):
        if global_feat.dim() == 2:
            global_feat = global_feat.unsqueeze(1)           # [B, 1, D]
        out, _ = self.attn(query=spot_feat, key=global_feat, value=global_feat)
        return self.norm(spot_feat + out)                    # Pre-LN residual

## 4. VISTA Main Model (ImgtoGene)

In [ ]:
class ImgtoGene(nn.Module):
    """
    VISTA: Vision-Spatial-Transcriptomic Analysis.
    Pipeline: MultiScaleConvNeXt → ViT + pos embed → GAT (img) → TITAN global
           → GAT (UNI2) → Cross-Attn → Fusion → MLP → gene expression.
    """
    def __init__(self, n_genes=1000):
        super(ImgtoGene, self).__init__()
        self.CBAMConv = ImprovedMultiScaleConvNeXt(in_channels=3, out_channels=32)
        self.x_embed = nn.Embedding(64, 1024)
        self.y_embed = nn.Embedding(64, 1024)
        self.vit = ViT()
        self.gat = MultiHeadGAT(in_features=1024, nhid=1024, out_features=512,
                                heads=8, dropout=0.2, alpha=0.01)
        self.gat1 = MultiHeadGAT(in_features=512, nhid=1024, out_features=512,
                                 heads=8, dropout=0.2, alpha=0.01)
        self.cross_attn = GlobalToLocalCrossAttention(dim=512, num_heads=8)
        self.AttentionAggregationLayer = AttentionLayer(512, 512)
        self.cemb_proj = nn.Sequential(
            nn.Linear(1536, 512), nn.SiLU(), nn.Linear(512, 512))
        self.cemb_proj2 = nn.Sequential(
            nn.Linear(1536, 512), nn.SiLU(), nn.Linear(512, 768))
        self.cemb_proj3 = nn.Sequential(
            nn.Linear(768, 512), nn.SiLU(), nn.Linear(512, 512))
        self.mlp = nn.Sequential(
            nn.Linear(512, 512 * 4), nn.SiLU(), nn.LayerNorm(512 * 4),
            nn.Dropout(0.2), nn.Linear(512 * 4, 512), nn.SiLU(),
            nn.LayerNorm(512), nn.Dropout(0.1), nn.Linear(512, n_genes))
        self.init_weights()

    def init_weights(self):
        """Initialize linear layers with truncated normal distribution."""
        for module in [self.mlp, self.cemb_proj, self.cemb_proj2, self.cemb_proj3]:
            for layer in module:
                if isinstance(layer, nn.Linear):
                    size = layer.weight.size()
                    fan_out, fan_in = size[0], size[1]
                    std = np.sqrt(2.0 / (fan_in + fan_out))
                    layer.weight.data.normal_(0.0, std)
                    layer.bias.data.normal_(0.0, 0.001)

    def forward(self, patches, features, adj, center, positions):
        B, N, C, H, W = patches.shape
        patches = patches.reshape(B * N, C, H, W)

        # 1. Multi-scale ConvNeXt
        patches = self.CBAMConv(patches)

        # 2. Position encoding
        positions = positions.long()
        centers_x = self.x_embed(positions[:, :, 0]).permute(1, 0, 2).squeeze(1)
        centers_y = self.y_embed(positions[:, :, 1]).permute(1, 0, 2).squeeze(1)
        ct = centers_x.unsqueeze(0) + centers_y.unsqueeze(0)

        # 3. ViT
        x = self.vit(patches, ct)                            # [N, 1024]

        # 4. Image GAT
        cemb_f1 = self.gat(x, adj)                           # [N, 512]

        # 5. UNI2 feature projection
        B, N, F = features.shape
        features = features.reshape(B * N, F)
        cemb = self.cemb_proj(features)                      # [N, 512]

        # 6. TITAN global slide encoding
        H0_features = self.cemb_proj2(features)              # [N, 768]
        center = center.squeeze(0).long()
        patch_size_lv0 = 112
        H0_features = H0_features.unsqueeze(0)
        with torch.autocast('cuda', torch.float16), torch.inference_mode():
            slide_embedding2 = titan.encode_slide_from_patch_features(
                H0_features, center, patch_size_lv0)         # [1, 768]
        global_feat = slide_embedding2.clone().float()
        global_feat = self.cemb_proj3(global_feat).squeeze() # [512]

        # 7. UNI2 GAT
        cemb1 = self.gat1(cemb, adj)                         # [N, 512]

        # 8. Global-to-local cross-attention
        cemb_ca = self.cross_attn(cemb1, global_feat.unsqueeze(0))

        # 9. Multi-modal fusion
        cemb_f3 = self.AttentionAggregationLayer(
            cemb_f1.squeeze(0), cemb_ca.squeeze(0))

        # 10. Prediction MLP
        output = self.mlp(cemb_f3)
        return output.squeeze()

## 5. Load TITAN, Set Up Loss & Optimizer

In [ ]:
titan = AutoModel.from_pretrained("./TITAN", trust_remote_code=True, local_files_only=True)
titan = titan.to(device)

def correlation_loss(x, y):
    """Loss = 1 - mean(Pearson r) across genes. Encourages high correlation."""
    mu_x = torch.mean(x, dim=0, keepdim=True)
    mu_y = torch.mean(y, dim=0, keepdim=True)
    xm, ym = x - mu_x, y - mu_y
    r_num = torch.sum(xm * ym, dim=0)
    r_den = torch.norm(xm, p=2, dim=0) * torch.norm(ym, p=2, dim=0)
    r_val = r_num / (r_den + 1e-8)
    r_val = torch.nan_to_num(r_val, nan=-1.0)
    return 1 - torch.mean(r_val)

log_file = "training_log0.txt"
MSE_loss = nn.MSELoss()

model = ImgtoGene(n_genes=785)
model.to(device)
optimizer = Adam(model.parameters(), lr=learning_rate)

## 6. Training Loop (LOO-CV, 1000 epochs)

In [ ]:
train_dataset = Her2ST(train=True, fold=fold)
test_dataset = Her2ST(train=False, fold=fold)

train_loader = DataLoader(train_dataset, batch_size=1, num_workers=2,
                          pin_memory=True, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, num_workers=2,
                         pin_memory=True, shuffle=False)

best_corr = -1

for epoch in range(1000):
    model.train()
    train_loss = 0

    for patch, feature, exp, adj, center, position in train_loader:
        patch, feature, exp, adj, center, position = [
            x.to(device) for x in [patch, feature, exp, adj, center, position]]
        exp = exp.squeeze()

        recon_gene = model(patch, feature, adj, center, position)
        loss_mse = MSE_loss(recon_gene, exp)
        loss_cor = correlation_loss(recon_gene, exp)
        loss = loss_mse + loss_cor

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    if (epoch + 1) % 2 == 0:
        corr_mean1, corr_mean2 = predict_and_evaluate(model, test_loader, device=device)
        log_message = (f"epoch {epoch+1}, Spot corr: {corr_mean1:.4f}, Gene corr: {corr_mean2:.4f}")
        print(log_message)
        with open(log_file, 'a') as f:
            f.write(f"{log_message}\n")
        if corr_mean1 + corr_mean2 > best_corr:
            best_corr = corr_mean1 + corr_mean2
            torch.save(model.state_dict(), f"model/Benchmarking_breasttest_{fold}.ckpt")
            print(f"New best model at epoch {epoch+1}")

## 7. Evaluation & Visualization (commented template)

In [ ]:
# # Load best model and predict on test sample
# model = ImgtoGene.load_from_checkpoint(
#     f"model/Benchmarking_breasttest_{fold}.ckpt", n_genes=785, learning_rate=1e-5)
# adata_pred, adata_truth = model_predict(model, test_loader, device=device)
#
# # Visualize predicted vs true gene expression on tissue
# import matplotlib.pyplot as plt
# im = Image.open('./对比方法/THItoGene-main/data/her2st/data/ST-imgs/A/A1/9769_C1_HE_small.jpg')
# plt.scatter(adata_pred.obs['pixel_x'], adata_pred.obs['pixel_y'],
#             c=adata_pred[:, 'MZT2A'].X, cmap='viridis', marker='o', s=1)
# plt.imshow(im); plt.title('Predicted MZT2A'); plt.show()
#
# plt.scatter(adata_truth.obs['pixel_x'], adata_truth.obs['pixel_y'],
#             c=adata_truth[:, 'MZT2A'].X, cmap='viridis', marker='o', s=1)
# plt.imshow(im); plt.title('True MZT2A'); plt.show()
#
# # Compute per-gene and per-spot Pearson correlation
# Percor_score_gene, mean_gene = cal_Percor(adata_truth, adata_pred)
# Percor_score_spot, mean_spot = cal_Percor(adata_truth, adata_pred)
# print(f"Mean gene-level corr: {mean_gene:.4f}")
# print(f"Mean spot-level corr: {mean_spot:.4f}")
#
# # Save results
# adata_truth.write(f'./experiments/her2st_groundtruth_{fold}.h5ad')
# adata_pred.write(f'./experiments/her2st_prediction_{fold}.h5ad')